In [ ]:
import json
import os
import pickle
import argparse
from pathlib import Path
from typing import Optional

import numpy as np
from sklearn.datasets import fetch_california_housing, load_wine
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

# Handle Colab/Notebook specific imports
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

# Ensure XGBoost and Torch are available
try:
    import xgboost as xgb
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torchvision.transforms as transforms
    from torch.utils.data import DataLoader
    from torchvision.datasets import CIFAR10
    from torchvision.models import resnet18
except ImportError:
    print("Installing missing dependencies for Colab...")
    os.system("pip install xgboost torch torchvision tqdm")
    import xgboost as xgb
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torchvision.transforms as transforms
    from torch.utils.data import DataLoader
    from torchvision.datasets import CIFAR10
    from torchvision.models import resnet18

## --- Generator Functions ---


def generate_sklearn_checkpoints(
    output_dir: Path, n_checkpoints: int = 20, trees_per_checkpoint: int = 10
) -> None:
    sklearn_dir = output_dir / "sklearn"
    sklearn_dir.mkdir(parents=True, exist_ok=True)

    X, y = load_wine(return_X_y=True)
    X = StandardScaler().fit_transform(X)

    model = GradientBoostingClassifier(
        n_estimators=trees_per_checkpoint,
        warm_start=True,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        random_state=42,
    )

    for step in tqdm(range(1, n_checkpoints + 1), desc="Generating sklearn (Wine)"):
        total_trees = step * trees_per_checkpoint
        model.set_params(n_estimators=total_trees)
        model.fit(X, y)

        ckpt_path = sklearn_dir / f"sklearn_gb_step_{total_trees:06d}.pkl"
        with open(ckpt_path, "wb") as f:
            pickle.dump(model, f)

        meta = {
            "step": total_trees,
            "train_accuracy": model.score(X, y),
            "framework": "sklearn",
        }
        with open(ckpt_path.with_suffix(".json"), "w") as f:
            json.dump(meta, f, indent=2)


def generate_xgboost_checkpoints(
    output_dir: Path, total_rounds: int = 1000, checkpoint_interval: int = 100
) -> None:
    xgboost_dir = output_dir / "xgboost"
    xgboost_dir.mkdir(parents=True, exist_ok=True)

    X, y = fetch_california_housing(return_X_y=True)
    X = StandardScaler().fit_transform(X).astype(np.float32)
    y = y.astype(np.float32)

    dtrain = xgb.DMatrix(X, label=y)
    params = {
        "max_depth": 4,
        "eta": 0.05,
        "objective": "reg:squarederror",
        "seed": 42,
    }

    booster = None
    evals_result = {}
    checkpoints = range(checkpoint_interval, total_rounds + 1, checkpoint_interval)

    for target_round in tqdm(checkpoints, desc="Generating XGBoost (Housing)"):
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=checkpoint_interval,
            xgb_model=booster,
            evals=[(dtrain, "train")],
            evals_result=evals_result,
            verbose_eval=False,
        )

        ckpt_path = xgboost_dir / f"xgboost_rounds_{target_round:06d}.xgb"
        booster.save_model(str(ckpt_path))

        meta = {
            "step": target_round,
            "train_rmse": evals_result["train"]["rmse"][-1],
            "framework": "xgboost",
        }
        with open(ckpt_path.with_suffix(".json"), "w") as f:
            json.dump(meta, f, indent=2)


def generate_pytorch_checkpoints(
    output_dir: Path, n_epochs: int = 20, checkpoint_interval: int = 1
) -> None:
    pytorch_dir = output_dir / "pytorch"
    pytorch_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[pytorch] Training on {device}")

    transform_train = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ]
    )

    dataset = CIFAR10(
        root="./data", train=True, download=True, transform=transform_train
    )
    loader = DataLoader(
        dataset,
        batch_size=128,
        shuffle=True,
        num_workers=2 if device.type == "cuda" else 0,
    )

    model = resnet18(num_classes=10).to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[10, 15], gamma=0.1
    )
    criterion = nn.CrossEntropyLoss()

    for epoch in tqdm(range(1, n_epochs + 1), desc="Generating PyTorch (CIFAR10)"):
        model.train()
        epoch_loss, correct, total = 0.0, 0, 0

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        if epoch % checkpoint_interval == 0:
            ckpt_path = pytorch_dir / f"pytorch_resnet18_epoch_{epoch:06d}.pt"
            torch.save(model.state_dict(), ckpt_path)

            with open(ckpt_path.with_suffix(".json"), "w") as f:
                json.dump(
                    {
                        "step": epoch,
                        "train_acc": correct / total,
                        "learning_rate": scheduler.get_last_lr()[0],
                        "framework": "pytorch",
                    },
                    f,
                    indent=2,
                )


## --- Main Execution Block ---


def main():
    # Colab-friendly default paths
    BASE_DIR = Path("/content/benchmark_checkpoints")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Saving all checkpoints to: {BASE_DIR}")

    # 1. Scikit-Learn
    generate_sklearn_checkpoints(BASE_DIR, n_checkpoints=10, trees_per_checkpoint=10)

    # 2. XGBoost
    generate_xgboost_checkpoints(BASE_DIR, total_rounds=500, checkpoint_interval=50)

    # 3. PyTorch
    generate_pytorch_checkpoints(BASE_DIR, n_epochs=20, checkpoint_interval=1)

    print(f"\nGeneration complete. Check files in the sidebar under: {BASE_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
import io
import os
import sys
import json
import pickle
import platform
import argparse
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import zstandard as zstd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time  # Moved import for the time module to top


# --- Mocking benchmark.utils for Standalone Colab Run ---
# In a real repo, these would be separate files.
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.end = time.perf_counter()
        self.elapsed_ms = (self.end - self.start) * 1000


def chunk_tensors(arr: np.ndarray, chunk_size: int) -> Tuple[List[bytes], np.dtype]:
    raw = arr.tobytes()
    chunks = [raw[i : i + chunk_size] for i in range(0, len(raw), chunk_size)]
    return chunks, arr.dtype


def extract_tensors_pytorch(state_dict):
    return {k: v.cpu().numpy() for k, v in state_dict.items()}


def extract_tensors_sklearn(model):
    # Extract trees from GradientBoosting (each tree is an array of nodes)
    tensors = {}
    for i, stage in enumerate(model.estimators_):
        for j, tree in enumerate(stage):
            tensors[f"stage_{i}_tree_{j}_nodes"] = tree.tree_.value
    return tensors


def extract_tensors_xgboost(booster):
    # XGBoost doesn't expose weights as cleanly as PyTorch;
    # for Phase 0, we treat the serialized model chunk as the "tensor"
    return {"model_blob": np.frombuffer(booster.save_raw(), dtype=np.uint8)}


def load_pytorch_checkpoint(path):
    import torch

    return torch.load(path, map_location="cpu")


def load_sklearn_checkpoint(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def load_xgboost_checkpoint(path):
    import xgboost as xgb

    bst = xgb.Booster()
    bst.load_model(str(path))
    return bst


# --- Core Measurer Logic ---


class CheckpointPairMeasurer:
    def __init__(self, chunk_sizes=None, tau_values=None):
        self.chunk_sizes = chunk_sizes or [256 * 1024, 1024 * 1024, 4 * 1024 * 1024]
        self.tau_values = tau_values or [0.5, 0.7, 0.9]
        self._compressor = zstd.ZstdCompressor(level=3)

    def measure_pair(
        self, ckpt_a_path: Path, ckpt_b_path: Path, framework: str
    ) -> Dict[str, Any]:
        tensors_a = self._extract(ckpt_a_path, framework)
        tensors_b = self._extract(ckpt_b_path, framework)

        results = {
            "total_chunks_compared": 0,
            "total_identical_chunks": 0,
            "by_chunk_size": {},
        }

        for cs in self.chunk_sizes:
            cs_result = {
                "measurements": [],
                "restore_times_ms": [],
                "tau_stats": {t: {"accepted": 0, "total": 0} for t in self.tau_values},
            }

            for name, arr_b in tensors_b.items():
                if name not in tensors_a:
                    continue
                arr_a = tensors_a[name]

                # Align shapes if model grew (common in sklearn warm_start)
                if arr_a.shape != arr_b.shape:
                    m = min(arr_a.size, arr_b.size)
                    arr_a, arr_b = arr_a.ravel()[:m], arr_b.ravel()[:m]

                if np.array_equal(arr_a, arr_b):
                    results["total_identical_chunks"] += 1
                    continue

                chunks_a, dtype = chunk_tensors(arr_a, cs)
                chunks_b, _ = chunk_tensors(arr_b, cs)

                for b_a, b_b in zip(chunks_a, chunks_b):
                    if len(b_a) != len(b_b):
                        continue

                    # Measurement
                    v_a = np.frombuffer(b_a, dtype=dtype).astype(np.float64)
                    v_b = np.frombuffer(b_b, dtype=dtype).astype(np.float64)
                    delta = v_b - v_a

                    with Timer() as t:
                        _ = v_a + delta

                    comp_delta = self._compressor.compress(delta.tobytes())
                    ratio = len(comp_delta) / len(b_b)

                    cs_result["restore_times_ms"].append(t.elapsed_ms)
                    cs_result["measurements"].append({"delta_ratio": ratio})
                    for tau in self.tau_values:
                        cs_result["tau_stats"][tau]["total"] += 1
                        if ratio < tau:
                            cs_result["tau_stats"][tau]["accepted"] += 1
                    results["total_chunks_compared"] += 1

            results["by_chunk_size"][cs] = cs_result
        return results

    def _extract(self, path, framework):
        if framework == "sklearn":
            return extract_tensors_sklearn(load_sklearn_checkpoint(path))
        if framework == "xgboost":
            return extract_tensors_xgboost(load_xgboost_checkpoint(path))
        if framework == "pytorch":
            return extract_tensors_pytorch(load_pytorch_checkpoint(path))


# --- Utils and Plotting ---


def classify_training_phase(step, total):
    progress = step / total
    if progress < 0.2:
        return "early"
    if progress < 0.6:
        return "mid"
    return "late"


def plot_gate_metrics(all_results):
    plt.figure(figsize=(10, 5))
    for fw in set(r["framework"] for r in all_results):
        fw_data = [r for r in all_results if r["framework"] == fw]
        steps = [r["pair_index"] for r in fw_data]
        # Plotting the 1MB chunk size ratios
        ratios = [
            np.mean(
                [m["delta_ratio"] for m in r["by_chunk_size"][1048576]["measurements"]]
            )
            for r in fw_data
        ]
        plt.plot(steps, ratios, marker="o", label=f"{fw}")

    plt.axhline(y=0.7, color="r", linestyle="--", label="Target τ=0.7")
    plt.title("Phase 0: Delta Compressibility Evolution")
    plt.xlabel("Checkpoint Pair Index")
    plt.ylabel("Avg Delta Ratio (Lower = More Compressible)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def print_gate_table(all_results):
    print(
        f"\n{'Framework':<12} {'Phase':<8} {'Chunk':>8}  {'Avg Δ Ratio':>12}  {'Accept@τ=0.7':>13}  {'Verdict'}"
    )
    print("-" * 80)
    for row in all_results:
        for cs, data in row["by_chunk_size"].items():
            if not data["measurements"]:
                continue
            ratio = np.mean([m["delta_ratio"] for m in data["measurements"]])
            acc = data["tau_stats"][0.7]["accepted"] / data["tau_stats"][0.7]["total"]
            verdict = "STORE_DELTA ✓" if ratio < 0.7 and acc > 0.5 else "STORE_FULL"
            print(
                f"{row['framework']:<12} {row['phase']:<8} {cs // 1024:>6}KB  {ratio:>12.3f}  {acc:>12.1%}  {verdict}"
            )


# --- Execution ---


def main():
    CHECKPOINT_DIR = Path("/content/benchmark_checkpoints")
    # 1. Run Generators (Assuming available in env)
    # print("Generating Checkpoints...")
    # generate_pytorch_checkpoints(CHECKPOINT_DIR, n_epochs=5)
    # generate_xgboost_checkpoints(CHECKPOINT_DIR, total_rounds=100, checkpoint_interval=20)
    # generate_sklearn_checkpoints(CHECKPOINT_DIR, n_checkpoints=5)

    # 2. Measure
    measurer = CheckpointPairMeasurer()
    all_results = []

    # Define specific glob patterns for each framework
    FRAMEWORK_PATTERNS = {
        "sklearn": "*.pkl",
        "xgboost": "*.xgb",
        "pytorch": "*.pt",
    }

    for fw in ["sklearn", "xgboost", "pytorch"]:
        glob_pattern = FRAMEWORK_PATTERNS[fw]
        # Corrected globbing to target specific file types
        files = sorted((CHECKPOINT_DIR / fw).glob(glob_pattern))
        for i in range(len(files) - 1):
            res = measurer.measure_pair(files[i], files[i + 1], fw)
            res.update(
                {
                    "framework": fw,
                    "pair_index": i,
                    "phase": classify_training_phase(i, len(files) - 1),
                }
            )
            all_results.append(res)

    # 3. Report
    print_gate_table(all_results)
    plot_gate_metrics(all_results)


if __name__ == "__main__":
    main()